# Exploratory Data Analysis (EDA)

## Imports

In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from typing import List
from scipy.stats import norm
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

## Dataset

### Dataset path
Indicate here the path to your dataset.

In [ ]:
DATASET_PATH = '../../dataset'
SEQUENCES = [x for x in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, x))] # All sequences
# SEQUENCES = ['sequence1', 'sequence2', '...']  # Individual sequences

### Load the data (frames & labels)

In [3]:
def load_data(DATASET_PATH: str, sequences: List[str]):
    """Function that takes in a dataset path and a list of sequences, and
    returns a dictionary containing all the data.

    :param DATASET_PATH: path to a dataset.
    :param sequences: list of strings containing the name of the sequences.
    :return:
        data: dictionary containing all the data.
    """
    # Prepare dictionary to store all data in the form of:
    # data[sequence][frame] = {'frame_path': frame_path, 'bboxes': bboxes}
    data = {sequence: {} for sequence in sequences}

    for sequence in sequences:
        sequence_labels_path = os.path.join(DATASET_PATH, sequence, 'labels')

        for label_filename in sorted(os.listdir(sequence_labels_path)):
            # Load labels from the label file
            label_path = os.path.join(sequence_labels_path, label_filename)
            if os.path.isfile(label_path):
                frame_filename, bboxes = load_labels(label_path)

                # Get path to frame
                frame_path = os.path.join(DATASET_PATH, sequence, 'images', frame_filename)

                # Save the data into a nested dictionary
                frame_name = label_filename.split('.')[0]
                data[sequence][frame_name] = {'frame_path': frame_path, 'bboxes': bboxes}

    return data


def load_labels(xml_file: str):
    """Function that takes in a path to an .xml file, and extracts the labels
    and bboxes it contains.

    :param xml_file: path to an .xml file.
    :return:
        frame_file_name: name of the frame associated to the .xml file.
        all_bboxes: list that contains all the bboxes contained in the .xml file.
            Each bbox is represented as a list of four values [x_min, y_min, x_max, y_max].
    """
    # Load xml data
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Load frame name
    frame_filename = root.find('filename').text

    # Get all bounding boxes
    all_bboxes = []
    for bboxes in root.iter('object'):
        
        # Get corners
        x_min = int(bboxes.find("bndbox/xmin").text)
        y_min = int(bboxes.find("bndbox/ymin").text)
        x_max = int(bboxes.find("bndbox/xmax").text)
        y_max = int(bboxes.find("bndbox/ymax").text)

        # Create and save boxes
        bbox = [x_min, y_min, x_max, y_max]
        all_bboxes.append(bbox)

    return frame_filename, all_bboxes

In [4]:
DATA = load_data(DATASET_PATH, SEQUENCES)

### Show a single frame

In [5]:
def show_frame(element: dict, alpha: int = 0.5):
    """Function that takes in a dictionary containing the data associated to an
    frame and its labels, and displays this information.

    :param element: dictionary containing the data associated to a frame and its labels.
    :param alpha: transparency value of the bboxes displayed.
    :return:
        None
    """
    # Load frame
    frame = cv2.imread(element['frame_path'])

    # Load bboxes
    bboxes = element['bboxes']

    # Overlay bboxes onto the frame
    for box in bboxes:
        
        # Initialize blank mask frame of the same dimensions for drawing the shapes
        shapes = np.zeros_like(frame, np.uint8)

        # Draw shapes
        cv2.rectangle(
            shapes,
            (box[0], box[1]),   # p1(x_min, y_min)
            (box[2], box[3]),   # p2(x_max, y_max)
            (255, 255, 255),    # color
            cv2.FILLED,
        )

        # Generate output by blending frame with shapes frame, using the shapes frames
        # also as mask to limit the blending to those parts
        mask = shapes.astype(bool)
        frame[mask] = cv2.addWeighted(frame, alpha, shapes, 1 - alpha, 0)[mask]

    # Show frame
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.imshow(frame)

In [6]:
# Show a single frame
show_frame(DATA['RoboMaster-Aisle']['frame_0015'])

<Figure size 640x480 with 1 Axes>

## Display frame distribution across scenarios

In [7]:
def plot_frames_distribution(data):
    """Function that takes in a dictionary containing all the information associated
    to a dataset, and displays the distribution of frames across sequences/scenarios.

    :param data: dictionary containing all the information associated to a dataset.
    :return:
        None
    """
    # Extract scenario names and their respective frame counts
    scenarios = list(data.keys())
    frame_counts = [len(data[scenario]) for scenario in scenarios]

    # Create a bar plot
    ax = plt.bar(scenarios, frame_counts, width=0.4)  # color='blue'

    # Add annotations to the bars
    for p, count in zip(ax.patches, frame_counts):
        plt.text(p.get_x() + p.get_width()/4., p.get_y() + count + 1,   # (x, y)
                 str(count),                                            # text
                 fontsize=10, fontweight='bold', color='grey')          # style

    plt.xlabel('Scenarios')
    plt.ylabel('Number of frames')
    plt.title('Distribution of frames per Scenario')
    plt.xticks(rotation=45)
    plt.ylim([0, max(frame_counts)*1.1])
    plt.tight_layout()
    # plt.savefig('frame distribution.png')
    plt.show()

In [8]:
# Display the frame distribution for all scenarios
plot_frames_distribution(DATA)

<Figure size 640x480 with 1 Axes>

## Display the bbox distribution across scenarios

In [9]:
def plot_bboxes_distribution(data):
    """Function that takes in a dictionary containing all the information associated
    to a dataset, and displays the distribution of bboxes across sequences/scenarios.

    :param data: dictionary containing all the information associated to a dataset.
    :return:
        None
    """
    # Extract scenario names and their respective bboxes counts
    scenarios = list(data.keys())
    bboxes_counts = []
    for scenario in scenarios:
        count = 0
        for frame in data[scenario].keys():
            count += len(data[scenario][frame]['bboxes'])
        bboxes_counts.append(count)

    # Create a bar plot
    ax = plt.bar(scenarios, bboxes_counts, width=0.4)  # color='blue'

    # Add annotations to the bars
    for p, count in zip(ax.patches, bboxes_counts):
        plt.text(p.get_x() + p.get_width()/4., p.get_y() + count + 1,   # (x, y)
                 str(count),                                            # text
                 fontsize=10, fontweight='bold', color='grey')          # style

    plt.xlabel('Scenarios')
    plt.ylabel('Number of Bboxes')
    plt.title('Distribution of Bboxes per Scenario')
    plt.xticks(rotation=45)
    plt.ylim([0, max(bboxes_counts)*1.1])
    plt.tight_layout()
    # plt.savefig('Bbox distribution.png')
    plt.show()

In [10]:
# Display the bbox distribution for all scenarios
plot_bboxes_distribution(DATA)

<Figure size 640x480 with 1 Axes>

## Display the bbox area distribution across scenarios

In [11]:
def plot_bboxes_area_distribution(data: dict, bins: int = 15) -> None:
    """Function that takes in a dictionary containing all the information associated
    to a dataset, and displays the distribution of the area of bboxes across sequences/scenarios.

    :param data: dictionary containing all the information associated to a dataset.
    :param bins: number of bins to display in the histogram.
    :return:
        None
    """
    # Create a subplot with multiple histograms
    fig, axes = plt.subplots(nrows=1, ncols=len(data), figsize=(15, 5))

    # Define colors for each scenario
    colors = [
        'b', 'g', 'r', 'c', 'm',
        'tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple',
        'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan'
    ]

    # Add a title above all the subplots
    plt.suptitle('Distribution of Bboxes area per Scenario')

    for i, (scenario, frame_data) in enumerate(data.items()):
        # Calculate bounding box areas for the scenario
        bbox_areas = []
        for frame_info in frame_data.values():
            for bbox in frame_info['bboxes']:
                x_min, y_min, x_max, y_max = bbox
                area = (x_max - x_min) * (y_max - y_min) / 1_000    # divide by 1_0000 for shorter x labels
                bbox_areas.append(area)

        # Plot the histogram for the scenario
        axes[i].hist(bbox_areas, bins=bins, color=colors[i], alpha=0.7)
        axes[i].set_title(scenario)
        axes[i].set_xlabel('Bounding Box Area (x1.000)')
        axes[i].set_ylabel('Frequency')

    # Adjust the layout
    plt.tight_layout()
    # plt.savefig('Bbox area distribution.png')
    plt.show()

In [12]:
# Display the bbox area distribution for all scenarios
plot_bboxes_area_distribution(DATA, bins=30)

<Figure size 1500x500 with 5 Axes>

## Display the RGB channel distribution across scenarios

In [13]:
def plot_rgb_channel_distribution(data):
    """Function that takes in a dictionary containing all the information associated
    to a dataset, and displays the distribution of the RGB frame channels across sequences/scenarios.

    :param data: dictionary containing all the information associated to a dataset.
    :return:
        None
    """
    # Create a subplot with multiple Gaussian distributions
    fig, axes = plt.subplots(nrows=1, ncols=len(data), figsize=(15, 5))

    # Add a title above all the subplots
    plt.suptitle("RGB Channel Distributions (Gaussian)")

    for i, (scenario, frame_data) in enumerate(data.items()):
        red_values = []
        green_values = []
        blue_values = []

        print(f'Sequence {i+1} ({scenario}):')
        for frame_info in tqdm(frame_data.values()):
            frame_path = frame_info['frame_path']

            # Load the frame using OpenCV
            frame = cv2.imread(frame_path)

            if frame is not None:
                # Extract RGB channels
                red_channel = frame[:, :, 2].mean()
                green_channel = frame[:, :, 1].mean()
                blue_channel = frame[:, :, 0].mean()

                # Append pixel values to the respective lists
                red_values.extend([red_channel])
                green_values.extend([green_channel])
                blue_values.extend([blue_channel])

        # Compute mean and standard deviation for each channel
        red_mean, red_std = np.mean(red_values), np.std(red_values)
        green_mean, green_std = np.mean(green_values), np.std(green_values)
        blue_mean, blue_std = np.mean(blue_values), np.std(blue_values)

        # Plot Gaussian distributions for each channel
        x = np.linspace(0, 255, 256)
        red_pdf = norm.pdf(x, red_mean, red_std)
        green_pdf = norm.pdf(x, green_mean, green_std)
        blue_pdf = norm.pdf(x, blue_mean, blue_std)

        axes[i].plot(x, red_pdf, color='red', label='Red')
        axes[i].plot(x, green_pdf, color='green', label='Green')
        axes[i].plot(x, blue_pdf, color='blue', label='Blue')
        axes[i].set_title(scenario)
        axes[i].set_xlabel("Pixel Value")
        axes[i].set_ylabel("Probability Density")
        axes[i].legend()

    # Adjust the layout
    plt.tight_layout()
    # plt.savefig('frame RGB channel distribution.png')
    plt.show()

In [14]:
# Display the RGB channel distribution for all scenarios
plot_rgb_channel_distribution(DATA)

Sequence 1 (RoboMaster-Parking):


 14%|█▍        | 18/126 [00:00<00:00, 178.62it/s]

100%|██████████| 126/126 [00:00<00:00, 189.81it/s]


Sequence 2 (RoboMaster-Stairs):


100%|██████████| 108/108 [00:00<00:00, 217.95it/s]


...

Sequence 4 (RoboMaster-Aisle):


100%|██████████| 129/129 [00:00<00:00, 214.02it/s]


Sequence 5 (RoboMaster-GTI):


100%|██████████| 122/122 [00:00<00:00, 209.52it/s]


<Figure size 1500x500 with 5 Axes>